# <center>大模型RAG入门基础架构介绍

## 一、传统大模型的局限性


在大模型真正落地使用时会发现，通用大模型基本无法满足我们真实的业务场景需求，主要有以下几个原因：

+ **知识可能过时（训练数据有时效性）**
    - 每个大语言模型的训练数据都是存在时效性的，所有的训练数据都是离线整理归纳完成的，所以在模型训练完成发布以后，之后发生的事情模型本身没有进行训练，也就无法回答这部分内容。

+ **<font color='red'>会产生"幻觉"（编造不存在的信息）</font>**
    - 大模型的"幻觉"是指模型生成的内容看似合理，但实际上与既定事实、真实数据或逻辑相违背，即"一本正经地胡说八道"。

+ **无法访问私有知识库数据**
    - 大模型本身学习不到企业或者个人的私有知识库知识，也就无法回答这部分的内容

+ **回答缺乏具体出处，难以验证**
    - 有些调研搜索的业务场景下，需要大模型对回答的内容给出具体的出处，一般是文章、论文等资料，这样可以进一步的核查模型回答内容的真实性。

+ **最大对话上下文限制（大部分模型128K）**
    - 大部分的模型上下文限制还是128k左右，一次性最大10万个汉字，可以处理的上下文长度是有限的

## 二、RAG的核心概念

### 1.什么是RAG？

**RAG** = **R**etrieval（检索） + **A**ugmented（增强） + **G**eneration（生成）

RAG即检索增强生成，为LLM提供了从某些数据源检索到的信息，并基于此修正生成的答案。RAG 基本上是Search + LLM 提示，可以通过大模型回答查询，并将搜索所找到的信息作为大模型的上下文。查询和检索到的上下文都会被注入到发送到 LLM 的提示语中。

**想象一下"开卷考试"**

+ **传统AI模型**：像"闭卷考试"——只能依靠模型本身记忆中的知识回答问题

+ **RAG系统**：像"开卷考试"——遇到回答不了的问题时，先去查阅资料（检索），然后结合用户的提问进行归纳整理（增强），最后LLM给出答案（生成）

一个简单完整的RAG系统如下图：用户进行了提问，提出的问题会先去知识库里进行检索答案，检索到相似度最高的前n个答案，然后和用户的提问一起放入到Prompt里，交给大语言模型，大语言模型根据用户的提问和给出检索的知识来进行整理总结，最终输出返回给用户结果。

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735623.png" width="1000">
</div>

### 2.RAG的本质：

**让大模型学会"查资料"后再回答问题**，而不是仅凭记忆回答。

+ 网络搜索资料

+ 知识库存储资料

这样既保证了答案的准确性、时效性，又提供了可追溯的信息来源！

### 3.RAG的优势？

+ **灵活性、适应性强**：可以接入最新、不断变化的数据
    - RAG的优势在于，能够接入最新的、私有的、还有不断更新的数据，来作为大模型外接知识库，来解决大模型回答问题的时效性问题

+ **提高准确性**：回答有据可查，减少幻觉
    - 基于知识库中检索到的知识来进行回答，回答的内容有据可查，也减少了模型的幻觉

+ **成本相对低**：无需重新训练或微调模型，只需维护知识库
    - 能够让大模型学会特定领域的知识灌入其实还有微调大模型这一方案，使用整理好的一定数量的数据集对LLM进行参数的调整，让大模型能够学习和理解我们自己的业务逻辑，有更好的zero-shot能力，但是需要算力成本，并且技术门槛比较高

+ **个性化程度高**：特别适合专业领域知识问答
    - 很多企业内部都在应用RAG技术，就是为了能够对接自己公司私有的知识数据，让用户能够直接对自己内部资料进行智能问答

### 4.真实使用案例

场景：咨询2025年最新的AI技术趋势

#### 传统AI的回答：

<div align="center">基于我2024年之前的训练数据，AI技术趋势包括...
                            （可能缺少2025年的最新信息）</div>


#### RAG系统的回答过程：

1. 检索到最新的行业报告、技术博客

2. 找到与"2025 AI趋势"最相关的段落

3. 结合这些最新信息生成回答：


<center> 根据2025年最新的行业报告显示，当前主要趋势包括：<br> 1. 多模态大模型的发展...<br> 2. Agent技术的普及...<br> 【信息来源：XX科技2025年度报告】 </center>


#### 联网搜索功能的场景：

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735606.png" width="700">
</div>

**核心逻辑：预抓取与静态检索**

+ 当你使用DeepSeek时，它调用的并非你问题中提及的某个网站的实时状态。DeepSeek和其他大模型厂商一样，运行着庞大的网络爬虫系统，持续地从互联网上抓取海量网页内容，经过处理后存入其内部数据库。你得到的答案，正是模型基于这个数据库中的信息生成的。

+ 由于依赖爬虫的周期性抓取，在不开启"联网搜索"时获得的信息可能存在延迟。正如此前一个测试发现，对于一个全新的或非常冷门的网站，不开启联网搜索，模型很可能无法获取其内容，或者只能找到数据库中近似的旧信息


**"联网搜索"功能的作用：**

+ **关闭时**：模型仅从上述的**静态数据库**中检索信息。对于一些热门或存在已久的话题，数据库里很可能已有相关数据，因此它能直接给出带有引用的回答，让你感觉它"搜了网页"。

+ **开启时**：系统会**额外发起一次实时的网络搜索**，旨在获取尽可能最新的信息，以补充或验证静态数据库的内容。这对于查询新闻、实时股价等瞬息万变的信息至关重要。

## 三、企业RAG核心应用场景

### 1. 企业知识库与智能问答

**场景描述**：  
企业拥有大量内部文档（员工手册、产品文档、技术规范、会议纪要等），员工需要快速找到准确信息。

**实际案例**：

+ 新员工入职培训问答（新员工培训成本高）

+ 产品技术规格查询（文档分散在不同系统中）

+ 公司政策咨询（搜索效率低，关键词匹配不精准）

### 2. 专业客服与技术支持

**场景描述**：  
为客户提供准确、一致的技术支持和问题解答。

**RAG优势**：

+ 基于最新产品文档和解决方案库（依赖客服人员的记忆）

+ 提供标准化的准确回答（回答不一致，依赖个人经验）

+ 减少培训时间，提高效率（处理复杂问题时响应慢）

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735594.png" width="700">
</div>

### 3. 科研与学术研究

**场景描述**：  
研究人员需要快速了解某个领域的最新进展和相关文献。

**价值体现**：

+ 快速文献调研

+ 跨领域知识整合

+ 研究趋势分析

还有其他的电商、法律、医疗、教育等等都是RAG应用的核心场景。

## 四、RAG的工作原理

一个最基础的RAG技术实现流程如下图所示：

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735382.png" width="700">
</div>

### 通用RAG基本工作流程（两阶段过程）

#### **阶段一：准备阶段（建立知识库）**

1. **数据接入**：收集各种文档（PDF、Word、网页等）

2. **文档解析**：提取文本内容

3. **文档分割**：将长文档切分成小片段

4. **向量化**：将文本转换为数学向量

5. **存储**：将向量存入专门的数据库


#### **阶段二：问答阶段（智能应答）**

1. **用户提问**：输入问题

2. **问题向量化**：将问题也转换成向量

3. **相似度检索**：在向量数据库中寻找最相关的文档片段

4. **构建增强提示**：将检索到的文档+原始问题组合成新的提示

5. **生成答案**：大语言模型基于增强后的提示生成最终答案

## 五、RAG完整执行流程详解

### 知识库构建阶段（离线）



#### 1. 数据源接入

**目标**：从各种来源收集原始数据

**常见数据源**：

+ 本地文件（PDF、Word、TXT、Markdown）

+ 数据库记录

+ 网页内容

+ API接口数据

+ 企业内部文档

In [ ]:
# 示例：从多种数据源接入
data_sources = {
    "本地文件": [".pdf", ".docx", ".txt"],
    "网页内容": ["博客", "文档", "新闻"],
    "数据库": ["MySQL", "MongoDB"],
    "API接口": ["企业知识库", "云存储"]
}

#### 2. 文档解析

**目标**：从原始文件中提取纯文本内容

**处理类型**：

+ PDF → 提取文字、表格

+ HTML → 去除标签，保留正文

+ Word → 解析文档结构

+ 图片 → OCR文字识别

**技术工具**：

+ PyPDF2、pdfplumber（PDF解析）

+ BeautifulSoup（HTML解析）

+ python-docx（Word解析）

+ Tesseract（OCR）

安装Langchain依赖包

**python版本必须是3.9或3.10**

In [2]:
!pip install langchain langchain-openai langchain-community langchain-huggingface langchain-text-splitters docx2txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [10]:
# 使用LangChain进行Word文档解析示例
from langchain_community.document_loaders import Docx2txtLoader
from google.colab import drive
drive.mount('/content/drive')

# 初始化加载器，传入Word文档路径，加载文档为Document对象
loader = Docx2txtLoader("/content/drive/MyDrive/企业报销制度.docx")
documents = loader.load()

# 查看加载结果
print(f"加载了 {len(documents)} 个文档片段")
print("----------------------------------")
print(f"第一段内容预览: {documents[0].page_content[:200]}...")
print("----------------------------------")
print(f"元数据: {documents[0].metadata}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
加载了 1 个文档片段
----------------------------------
第一段内容预览: 企业报销制度



1. 总则



1.1 本制度适用于所有正式员工，旨在确保公司资金的合理使用，提高报销流程的规范性、透明度及效率。



1.2 所有报销申请必须通过飞书完成，并在飞书指定流程中提交。任何通过其他方式提交的报销申请不予受理。



1.3 报销时间点及要求详见各类报销场景。



2. 报销范围



本公司报销包括但不限于以下场景：

   - 加班报销

   - 出差报...
----------------------------------
元数据: {'source': '/content/drive/MyDrive/企业报销制度.docx'}


#### 3. 文档分割

**目标**：将长文档切分成适合处理的知识片段（chunks）

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735381.png" width="1000">
</div>

**为什么需要分割？**

+ 大语言模型有输入长度限制

+ 提高检索精度

+ 避免信息过载

**常用分割方法**：

+ 固定长度分割

+ 按段落/句子分割

+ 语义分割（按主题变化）

In [12]:
!pip install -qU langchain-text-splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建文本分割器，按照固定长度切分，并保留重叠部分
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每个文本块的最大字符数
    chunk_overlap=100,   # 相邻文本块之间的重叠字符数（保持上下文连贯性）
    separators=["\n\n", "\n", "。", "！", "？", "，", ""],  # 定义了分割符的优先级顺序
    add_start_index=True,  # 记录每个起始位块在原文档中的置
)

# 执行分割
all_splits = text_splitter.split_documents(documents)
print(f"分割后得到 {len(all_splits)} 个文本块")
print(all_splits[0].page_content)
print("=====")
print(all_splits[1].page_content)

分割后得到 5 个文本块
企业报销制度



1. 总则



1.1 本制度适用于所有正式员工，旨在确保公司资金的合理使用，提高报销流程的规范性、透明度及效率。



1.2 所有报销申请必须通过飞书完成，并在飞书指定流程中提交。任何通过其他方式提交的报销申请不予受理。



1.3 报销时间点及要求详见各类报销场景。



2. 报销范围



本公司报销包括但不限于以下场景：

   - 加班报销

   - 出差报销（交通、住宿、日常补助等）

   - 住宿报销（适用于公司安排的住宿）

   - 商务宴请报销

   - 培训报销

   - 其他因公支出报销



3. 报销流程



所有报销流程通过飞书进行，具体流程如下：



3.1 **提交申请**

   - 员工通过飞书的“费用报销”模块，填写报销申请表。

   - 提交时需选择报销类型，并提供必要的证明文件，如发票、行程单、餐饮单据等。

   - 如因特殊原因需提前报销或事后补办，请在备注中注明原因。



3.2 **审批流程**

   - **直接主管审批**：所有报销申请先由员工的直接主管审核。
=====
- 如因特殊原因需提前报销或事后补办，请在备注中注明原因。



3.2 **审批流程**

   - **直接主管审批**：所有报销申请先由员工的直接主管审核。

   - **财务审批**：主管审核后，申请进入财务审批环节，财务将对申请内容、金额、发票等进行核实。

   - **高层审批**：超过一定金额（例如5000元）或涉及高额宴请、培训等支出的报销，需经相关部门负责人及财务总监或总经理审批。



3.3 **支付流程**

   - 报销申请审核通过后，财务部门会在每月固定日期（如每月5号、20号）进行统一支付。

   - 支付方式为银行转账，员工需在飞书个人资料中填写准确的银行账户信息。



4. 各类报销细则



4.1 加班报销



4.1.1 报销标准：依据公司加班补贴政策，加班可按餐费、交通费等标准进行报销。



4.1.2 申请流程：

   - 填写“加班报销申请表”，附上加班申请单及具体消费凭证。

   - 若加班时间超过22:00，可申请出租车费报销，须提供电子发票。



4.1.3 审批流程：部门主管审批后进入财务审批环节。

#### 4. 词向量化

在数学中，向量（也称为欧几里得向量、几何向量），指具有大小（magnitude）和方向的量。它可以形象化地表示为带箭头的线段。箭头所指：代表向量的方向；线段长度：代表向量的大小。

例如，二维空间中的向量可以表示为 (𝑥,𝑦)，表示从原点 (0,0) 到点 (𝑥,𝑦) 的有向线段

<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735386.png" width="1000">
</div>


+ 1.  将文本转成一组向量：每个下标 `i`，代表一个维度
+ 2.  整个数组对应一个 `n` 维空间的一个点，即**文本向量**又叫 Embeddings
+ 3.  向量之间可以计算距离，距离远近对应**语义相似度**大小

<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735388.png" width="1000">
</div>

**目标**：将文本转换为数值向量（数学表示）

**向量化原理**：

<center> 文本："RAG技术很强大" <br> ↓ 向量化模型（Embedding模型）<br> 向量：[0.12, -0.45, 0.78, ..., 0.33] (384维向量) </center>

**常用模型**：

+ OpenAI text-embedding-3

+ 国产模型：M3E、BGE等

安装依赖

In [15]:
!pip install openai sentence-transformers dotenv

**使用OpenAI 模型进行Embedding向量化**

In [17]:
#加载api_key环境
from dotenv import load_dotenv
import os

#加载环境变量
load_dotenv()

# OpenAI API 配置
OPENAI_API_KEY: str = os.getenv('OPENAI_API_KEY')
OPENAI_BASE_URL: str = os.getenv('OPENAI_BASE_URL')

# 默认模型配置
model_name = "text-embedding-3-small"

In [19]:
from openai import OpenAI
client = OpenAI(base_url='https://api.suyu.io/v1', api_key='sk-IOJHgSR9Ti91FtImRLGK6bSrXF36u1taIi7BSt7PvaNl175a')

# 准备需要向量化的文本
text_to_embed = "今天天气真好，我们出去散步吧！"

# 调用 Embedding 模型
response = client.embeddings.create(
    model=model_name,  # 使用默认模型配置中的模型名称
    input=text_to_embed # 需要向量化的文本
)

# 从响应中提取生成的向量
embedding_vector = response.data[0].embedding
print(f"生成向量的维度：{len(embedding_vector)}")
print(f"向量预览：{embedding_vector[:5]}...")

生成向量的维度：1536
向量预览：[0.03212086483836174, -0.002328116912394762, -0.025515884160995483, -0.014778180047869682, 0.03699157387018204]...


**使用国产模型进行Embedding向量化**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 初始化BGE嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-zh-v1.5',
    model_kwargs={'device': 'cuda'} # 使用GPU可改为 'cuda'
)

# 2. 准备示例文本
text = "机器学习是人工智能的重要分支"

# 3. 执行向量化
vector = embeddings.embed_documents([text])[0]

# 4. 打印结果
print(f"文本: '{text}'")
print(f"向量维度: {len(vector)}")
print(f"向量值: {vector[:5]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

如果运行时遇到下面的报错，只需要执行：pip install --upgrade ipywidgets

<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015150612096.png" width="1000">
</div>

#### 5. 文档存储

**目标**：将向量和原文存储到专用数据库中

**向量数据库选择**：

+ ChromaDB（轻量级，易用）www.trychroma.com/

+ Pinecone（云服务，高性能）www.pinecone.io/

+ Faiss（高并发，gpu加速）https://github.com/facebookresearch/faiss

+ Milvus（毫秒级搜索万亿级向量数据）http://milvus.io/

安装依赖

pip install faiss-cpu

有cuda的可以安装faiss-gpu：pip install faiss-gpu


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# 初始化嵌入模型
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-zh-v1.5',
    model_kwargs={'device': 'cpu'}  # 使用GPU可改为 'cuda'
)

# 创建向量数据库
vectorstore = FAISS.from_documents(all_splits, embeddings)

# 保存到本地（可选）
vectorstore.save_local("word_doc_faiss_index")

### 问答推理阶段（在线）

#### 1. 用户提问

用户输入自然语言问题：

"RAG技术的主要优势有哪些？"

#### 2. 文档检索

**过程**：

1. 将用户提出的问题向量化

2. 在向量数据库中搜索相似度最高的前N个文档

3. 返回最相关的文档片段

**检索算法**：

+ 余弦相似度

+ 欧氏距离

+ 近似最近邻搜索


#### 向量间的相似度计算


<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735379.png" width="1000">
</div>

<dev align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735384.png" width="500">
</div>

安装依赖包

pip install numpy

In [ ]:
from openai import OpenAI
import os
import numpy as np
#NumPy 库中的线性代数模块
import numpy.linalg as norm
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

def cos_sim(a,b):
    """计算余弦相似度，数值越大越相似"""
    return np.dot(a, b) / (norm.norm(a) * norm.norm(b))

def l2_dist(a,b):
    """计算欧氏距离，数值越小越相似"""
    return norm.norm(np.asarray(a) - np.asarray(b))

def get_embedding(texts, model="text-embedding-3-small"):
    """
    获取文本的 embedding 向量

    参数:
        texts: 字符串或字符串列表
        model: 模型名称

    返回:
        如果 texts 是单个字符串，返回单个向量（列表）
        如果 texts 是字符串列表，返回向量列表（列表的列表）
    """
    # 确保 texts 是列表格式
    if isinstance(texts, str):
        texts = [texts]

    response = client.embeddings.create(
        model=model,
        input=texts
    )

    # 返回所有文档的向量
    embeddings = [item.embedding for item in response.data]

    # 如果只有一个文本，返回单个向量；否则返回向量列表
    return embeddings[0] if len(embeddings) == 1 else embeddings

In [ ]:
# 支持跨语言计算
query = "Artificial Intelligence in Education"
# query = "人工智能在教育中的应用"
documents = [
    "机器学习算法正在改变传统教学模式，实现个性化学习路径推荐",
    "深度学习技术在自然语言处理领域的突破，使得智能辅导系统更加精准",
    "计算机视觉技术帮助在线教育平台实现学生专注度监测和行为分析",
    "大数据分析为教育决策提供数据支持，优化课程设计和教学资源配置",
    "虚拟现实和增强现实技术创造沉浸式学习体验，提高学习效果和参与度",
]

# 进行转换 Embedding 向量
# query 是单个字符串，get_embedding 返回单个向量
query_vec = get_embedding(query)

# documents 是列表，get_embedding 返回向量列表
doc_vecs = get_embedding(documents)

print(f"查询向量维度: {len(query_vec)}")
print(f"文档数量: {len(doc_vecs)}")
print(f"每个文档向量维度: {len(doc_vecs[0])}")

查询向量维度: 1536
文档数量: 5
每个文档向量维度: 1536


In [ ]:
print("余弦相似度计算:")
print("query_vec 与 query_vec 之间的余弦相似度:")
print(cos_sim(query_vec, query_vec))
print("---------------------------")
for i, vec in enumerate(doc_vecs):
    cos_distance = cos_sim(query_vec, vec)
    print(f"文档 {i+1}: {cos_distance:.4f}")

print("\n欧氏距离计算:")
print("query_vec 与 query_vec 之间的欧氏距离:")
print(l2_dist(query_vec, query_vec))
print("---------------------------")
for i, vec in enumerate(doc_vecs):
    distance = l2_dist(query_vec, vec)
    print(f"文档 {i+1}: {distance:.4f}")

余弦相似度计算:
query_vec 与 query_vec 之间的余弦相似度:
1.0
---------------------------
文档 1: 0.4551
文档 2: 0.4688
文档 3: 0.4475
文档 4: 0.4349
文档 5: 0.4576

欧氏距离计算:
query_vec 与 query_vec 之间的欧氏距离:
0.0
---------------------------
文档 1: 1.0439
文档 2: 1.0308
文档 3: 1.0511
文档 4: 1.0631
文档 5: 1.0416


##### 向量数据库搜索查询文档

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 加载已保存的FAISS向量数据库
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-zh-v1.5',
    model_kwargs={'device': 'cpu'}
)

vectorstore = FAISS.load_local(
    "word_doc_faiss_index",  # 你之前保存的目录名
    embeddings,
    allow_dangerous_deserialization=True  # 必要的安全警告确认机制，明确告诉用户"反序列化可能执行任意代码"
    # LangChain团队为了保护用户，默认禁止加载可能不安全的序列化数据，除非用户明确确认风险
)

# 2. 执行相似度搜索
query = "调休的申请流程是什么？"

#docs = vectorstore.similarity_search(query, k=3)  # 返回最相关的3个文档
# print(f"查询: '{query}'")
# print(f"找到 {len(docs)} 个相关文档:\n")
#
# for i, doc in enumerate(docs):
#     print(f"--- 结果 {i+1} ---")
#     print(f"内容: {doc.page_content}")
#     print(f"来源: {doc.metadata.get('source', 'Unknown')}")
#     print()

docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)  # 带相似度分数的搜索

# 3. 打印检索结果
for doc, score in docs_with_scores:
    print(f"相似度: {score:.4f} - 内容: {doc.page_content}...")
    print("==================================================")

相似度: 0.8162 - 内容: 有效期：加班工时的调休应在6个月内使用，未使用视为自动放弃。

劳动法一致性：调休制度符合劳动法，公司额外允许部分部门灵活安排调休为公司福利。

3. 假期审批流程

3.1 所有假期申请须在飞书的“请假管理”模块填写，提交后按照不同假期类型进行审批。
3.2 假期在部门主管及人力资源部批准后方可生效，任何未获批的假期视为旷工。

4. 其他规定

4.1 员工的请假记录将被记录在飞书系统中，以供考核及年终总结使用。
4.2 公司会在每季度对假期情况进行审核，以确保假期政策的公平性与合理性。...
相似度: 0.9476 - 内容: 公司假期制度



1. 总则

1.1 本公司依据国家劳动法为员工提供病假、事假、年假、婚假、调休等假期。
1.2 员工应合理安排请假时间，如遇特殊情况，应尽早通知并按规定申请假期。
1.3 所有假期的申请和审批流程均通过飞书完成，任何线下提交的假期申请将不予受理。

2. 各类假期及申请流程

2.1 事假

定义：因个人事务需要请假的时间。

申请流程：

在飞书中填写“事假申请”表，注明请假原因、起止日期及联系人。

事假须提前1天申请，并由部门主管审批。

工资扣除：事假为无薪假期，请假期间按实际天数扣除工资。

有效期：员工每年最多可申请10天事假，超过天数需经部门经理审批。

劳动法一致性：事假并非法定带薪假期。

2.2 病假

定义：因身体健康问题无法工作的休假时间。

申请流程：

短期病假（3天以内）：在飞书中填写“病假申请”表，并附上病假证明。

长期病假（超过3天）：需提供医院出具的详细病假条和病历复印件。

所有病假均需部门主管及人力资源部审批。

工资扣除：

	员工病假期间工资按国家规定支付病假工资，一般为不低于基本工资的80%。...
相似度: 0.9782 - 内容: 2.6 产假及陪产假

定义：女员工生育享有的产假和男员工的陪产假。

产假天数：依据劳动法规定，女员工享有98天产假，难产可增加15天，多胞胎每多一胎增加15天。

陪产假天数：符合劳动法规定的员工享有10天陪产假。

申请流程：

	产假：女员工需在飞书提交“产假申请”表，并附上医院产前检查证明。

	陪产假：男员工需提供结婚证及出生证明复印件。

工资扣除：产假和陪产假均为带薪假期，不扣除工资。

#### 3. 提示增强

**目标**：将检索到的文档与用户问题组合成增强提示

**提示模板示例**：

```plain
基于以下上下文信息，请回答问题：

【相关文档1】
RAG技术可以接入最新数据，减少模型幻觉...

【相关文档2】
RAG系统能够提供可验证的信息来源...

【相关文档3】
通过检索增强，模型可以回答专业领域问题...

问题：调休的申请流程是什么？
请根据上述上下文回答：
```

In [ ]:
def build_rag_prompt(query, retrieved_docs):
    """
    构建RAG提示词
    """
    # 1. 拼接检索到的文档内容
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 2. 构建提示词模板
    prompt = f"""请根据以下上下文信息回答问题。如果上下文中有答案，请基于上下文回答；如果没有，请说明。

        上下文信息：
        {context}

        问题：{query}

        请给出详细、准确的回答：
    """
    return prompt

#### 4. 答案生成

**目标**：大语言模型基于增强提示生成最终答案

**生成过程**：

增强提示 → 大语言模型 → 生成答案

**模型选择**：

+ GPT系列

+ 文心一言、通义千问

+ Llama、ChatGLM等开源模型

In [ ]:
def call_llm_with_rag(query, vectorstore, llm_model):
    """
    完整的RAG流程：检索 + 提示词构建 + 调用大模型
    """
    # 1. 检索相关文档
    retrieved_docs = vectorstore.similarity_search(query, k=3)

    # 2. 构建提示词
    prompt = build_rag_prompt(query, retrieved_docs)

    # 3. 调用大模型
    response = llm_model.invoke(prompt)

    return response.content

In [ ]:
from langchain_openai import ChatOpenAI

# 初始化OpenAI模型
llm_model = ChatOpenAI(
    model="gpt-3.5-turbo",
    openai_api_key=OPENAI_API_KEY,  # 替换为你的API密钥
    temperature=0.7
)

# 使用示例
query = "调休的申请流程是什么？"
result = call_llm_with_rag(query, vectorstore, llm_model)
print("用户提问：" + query)
print("模型最终回答：" + result)

用户提问：调休的申请流程是什么？
模型最终回答：调休的申请流程包括在飞书中填写“调休申请”表，注明调休日期及调休时长，并提交给部门主管审批。


### 完整流程演示

In [ ]:
def build_rag_prompt(query, retrieved_docs):
    """
    构建RAG提示词
    """
    # 1. 拼接检索到的文档内容
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # 2. 构建提示词模板
    prompt = f"""请根据以下上下文信息回答问题。如果上下文中有答案，请基于上下文回答；如果没有，请说明。

        上下文信息：
        {context}

        问题：{query}

        请给出详细、准确的回答：
    """
    return prompt

In [ ]:
# 使用LangChain进行Word文档解析示例
from langchain_community.document_loaders import Docx2txtLoader
# 文档分割
from langchain.text_splitter import RecursiveCharacterTextSplitter
# 向量数据库
from langchain_community.vectorstores import FAISS
# 向量化Embedding
from langchain_huggingface import HuggingFaceEmbeddings

# 初始化加载器，传入Word文档路径，加载文档为Document对象
loader = Docx2txtLoader("公司假期制度.docx")
documents = loader.load()

# 创建文本分割器，按照固定长度切分，并保留重叠部分
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每个文本块的最大字符数
    chunk_overlap=100,   # 相邻文本块之间的重叠字符数（保持上下文连贯性）
    separators=["\n\n", "\n", "。", "！", "？", "，", ""],  # 定义了分割符的优先级顺序
    add_start_index=True,  # 记录每个块在原文档中的起始位置
)

# 执行分割
all_splits = text_splitter.split_documents(documents)

# 初始化Embedding模型
embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-large-zh-v1.5',
    model_kwargs={'device': 'cpu'}  # 使用GPU可改为 'cuda'
)

# 创建向量数据库
vectorstore = FAISS.from_documents(all_splits, embeddings)

# 保存到本地（可选）
vectorstore.save_local("word_doc_faiss_index")

# 加载本地向量数据库存储
vectorstore = FAISS.load_local(
    "word_doc_faiss_index",  # 你之前保存的目录名
    embeddings,
    allow_dangerous_deserialization=True  # 必要的安全警告确认机制，明确告诉用户"反序列化可能执行任意代码"
    # LangChain团队为了保护用户，默认禁止加载可能不安全的序列化数据，除非用户明确确认风险
)

# 用户提问
query = "调休的申请流程是什么？"

# 检索相关文档
retrieved_docs = vectorstore.similarity_search(query, k=3)

# 构建提示词
prompt = build_rag_prompt(query, retrieved_docs)

# 调用大模型回答内容
response = llm_model.invoke(prompt)

print("用户提问：" + query)
print("模型最终回答：" + response.content)

用户提问：调休的申请流程是什么？
模型最终回答：调休的申请流程是在飞书中填写“调休申请”表，注明调休日期及调休时长，并经部门主管审批后方可生效。


<div align="center">
<img src="https://zrj18330672592.oss-cn-beijing.aliyuncs.com/20251015134735612.png" width="600">
</div>

<div style="font-size:14px; line-height: 1.8;">
    <strong>特别注意：</strong>
    <ol>
        <li style="margin-bottom: 15px;">
            <strong>大模型个数和类别</strong>：整个RAG流程涉及到两类大模型
            <ul style="margin-top: 8px;">
                <li>用于转换成向量的Embedding模型（用户提问阶段和知识库入库阶段的模型必须保持一致）</li>
                <li>用于生成答案的LLM模型</li>
            </ul>
        </li>
        <li style="margin-bottom: 15px;">
            <strong>语言模型应用场景</strong>：LLM模型只负责最终将检索到的知识和用户问题进行整合后，回复用户
            <ul style="margin-top: 8px;">
                <li>知识库的数据搭建工作不涉及LLM模型</li>
            </ul>
        </li>
        <li style="margin-bottom: 15px;">
            <strong>RAG最终目的</strong>：这一系列操作都是为了整合好合适的prompt提示词，最终交给LLM
            <ul style="margin-top: 8px;">
                <li>prompt是与LLM大模型唯一的交互方式</li>
            </ul>
        </li>
    </ol>
</div>

## 六、RAG具体实践落地策略

1. 手动搭建RAG引擎

2. 低代码平台搭建 （Coze、Dify、N8N）

3. 使用GLM、OpenAI Responses API 等进行快速实现

4. 使用LangChain、LlamaIndex等开源项目快速搭建

## 七、实际部署考虑

在选择RAG应用场景时，需要考虑：

1. **数据质量**：是否有高质量、结构化的知识源？

2. **更新频率**：知识需要多频繁更新？

3. **准确度要求**：错误的代价有多大？

4. **技术复杂度**：是否有相应的技术能力？

## 八、RAG应用的优化方案

想要真正进一步优化RAG的效果，需要考虑：

1. **优化文本分割**：尝试不同的文本分割器（如按句子、递归字符等）和块大小、重叠长度

2. **重排序（Re-ranking）**：使用重排序模型对检索到的文档进行重新排序，以提高顶部文档的相关性

3. **多轮对话与历史管理**：在对话中维护历史消息，并将历史信息纳入当前查询的上下文中

4. **混合检索**：结合关键词检索（如BM25）和向量检索，以及其他数据库，以提高检索的召回率

5. **评估与迭代**: 构建评估集，对RAG系统进行定量评估，从而指导优化方向